# 11 — Neural-network data preparation

**Owner:** Shaik Tajuddin · **Phase 2, Step 1**

Phase 1 used tree ensembles, which split on order and are therefore invariant to any
monotone transform of a feature. None of the decisions below mattered then.

A neural network is not invariant. It multiplies inputs by weights and sums them, so a
feature whose validation values reach 48 standard deviations dominates every unit it
feeds and saturates smooth activations.

This notebook decides four things, each from evidence measured on the **training
partition only**:

1. Which features need a transform, and which transform.
2. Standard or robust scaling.
3. How the upstream-imputation flags should be represented.
4. One-hot encoding or learned embeddings for the categoricals.

The work plan requires the Phase 1 pipeline to remain the default **unless a new one is
clearly better**, so the notebook ends with an MLP comparison rather than an argument.


In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')

from src.clean import MEASUREMENTS
from src.evaluate_api import get_splits
from src.dl.nn_preprocess import (
    skew_report, conditioning_table, compare_on_mlp, build_nn_preprocessor,
)

X_train, X_val, X_test, y_train, y_val, y_test = get_splits(
    include_metadata=True, scale_numeric=False)
print(f'train {len(X_train):,}   val {len(X_val):,}   test {len(X_test):,}')
print('features:', list(X_train.columns))


train 1,986,397   val 415,805   test 415,804
features: ['Ammonia (mg/l)', 'Biochemical Oxygen Demand (mg/l)', 'Dissolved Oxygen (mg/l)', 'Orthophosphate (mg/l)', 'pH (ph units)', 'Temperature (cel)', 'Nitrogen (mg/l)', 'Nitrate (mg/l)', 'Country', 'Waterbody Type']


## 1. Skew, and why the obvious transform does not work

The measurements are extremely right-skewed — ammonia reaches a skewness of 12.3 with
kurtosis 243. The textbook response is `log1p`.

**It is unusable on five of the eight columns**, because they contain negative values and
`log1p` is undefined below $-1$. Those negatives are not corrupt data: they are the
anomalies Phase 1 deliberately *flagged and retained* rather than deleting. The cleaning
decision propagates directly into the Phase 2 transform choice.

It is also actively harmful on the two columns that are **not** right-skewed. Dissolved
oxygen is left-skewed, and `log1p` makes it worse.


In [2]:
rows = []
for c in MEASUREMENTS:
    s = X_train[c]
    lg = np.log1p(s.clip(lower=0))
    rows.append({
        'feature': c, 'skew': round(s.skew(), 2), 'kurtosis': round(s.kurtosis(), 1),
        'min': round(s.min(), 3), 'negatives': int((s < 0).sum()),
        'log1p_valid': bool((s >= 0).all()), 'skew_if_log1p': round(lg.skew(), 2),
    })
skew_df = pd.DataFrame(rows)
display(skew_df)
print(f"columns with negative values: {int(skew_df.negatives.gt(0).sum())} of {len(skew_df)}")
print('log1p makes these WORSE:',
      [r.feature.split(' (')[0] for r in skew_df.itertuples()
       if abs(r.skew_if_log1p) > abs(r.skew)])


,feature,skew,kurtosis,min,negatives,log1p_valid,skew_if_log1p
0,Ammonia (mg/l),12.29,243.2,-0.005,2,False,3.37
1,Biochemical Oxygen Demand (mg/l),11.22,143.5,-2.000,1,False,2.60
2,Dissolved Oxygen (mg/l),-1.39,4.3,0.000,0,True,-3.17
3,Orthophosphate (mg/l),10.61,282.2,-0.003,1,False,2.71
4,pH (ph units),1.58,110.3,-1.000,1,False,-2.91
5,Temperature (cel),1.85,16.2,-1.120,19,False,-0.77
6,Nitrogen (mg/l),2.39,7.4,0.000,0,True,0.15
7,Nitrate (mg/l),6.13,78.9,0.000,0,True,0.23


columns with negative values: 5 of 8
log1p makes these WORSE: ['Dissolved Oxygen', 'pH']


### Yeo-Johnson instead

Yeo-Johnson is defined on negative values and is monotone, so the ordering of a
measurement is preserved. Fitted on train only.


In [3]:
rep = skew_report(X_train, 'yeo-johnson')
display(rep.to_frame())
print(f'mean |skew| before: {np.mean([abs(v) for v in rep.skew_before.values()]):.2f}')
print(f'mean |skew| after : {np.mean([abs(v) for v in rep.skew_after.values()]):.2f}')


,skew_before,skew_after,negative_values
Ammonia (mg/l),12.290,1.080,2
Biochemical Oxygen Demand (mg/l),11.223,-0.065,1
Dissolved Oxygen (mg/l),-1.388,0.323,0
Orthophosphate (mg/l),10.612,0.990,1
pH (ph units),1.576,1.024,1
Temperature (cel),1.850,0.089,19
Nitrogen (mg/l),2.393,0.024,0
Nitrate (mg/l),6.126,0.012,0


mean |skew| before: 5.93
mean |skew| after : 0.45


## 2. Standard or robust scaling

The usual advice with outliers present is `RobustScaler` (median and IQR). **On this data
it is dramatically worse**, and the reason is instructive: it divides by the IQR, and on a
long right tail the IQR is small, so the tail explodes rather than being tamed.

The deciding statistic for a neural network is how much of the validation input lands
outside roughly $[-5, 5]$, where smooth activations saturate. Fitted on train, measured on
validation.


In [4]:
cond = conditioning_table(X_train, X_val)
display(cond)
worst = cond.loc[cond.max_abs_value.idxmax()]
best  = cond.loc[cond.max_abs_value.idxmin()]
print(f"worst conditioned: {worst.strategy} (max |z| = {worst.max_abs_value})")
print(f"best  conditioned: {best.strategy} (max |z| = {best.max_abs_value})")


,strategy,pct_beyond_5sd,pct_beyond_10sd,max_abs_value,mean_abs_skew
0,phase1,0.7304,0.2208,48.04,6.328
1,robust,4.9781,3.0517,740.54,6.328
2,yeo-johnson,0.1572,0.0130,44.88,1.358
3,quantile,0.0274,0.0000,5.20,0.130


worst conditioned: robust (max |z| = 740.54)
best  conditioned: quantile (max |z| = 5.2)


## 3. The upstream-imputation flags in a neural network

Phase 1 found that five of eight measurement columns had been median-imputed before
publication, and added one boolean per affected column plus `n_imputed_upstream` (0–8).
They are **excluded from the primary feature set** because they proxy for `Country`, and
the Phase 1 ablation showed they move MAE by only 0.0043.

The question here is narrower and purely about representation: *if* a model uses them,
how should a network see them?

**Decision: pass binary flags through unscaled; scale the count.**

Standardising a 0/1 column rescales it by its prevalence in the training partition. The
"absent" state stops being zero, and the feature's magnitude becomes a function of how
common the flag happened to be — which interacts badly with weight decay (which shrinks
toward zero) and with dropout (which masks to zero). `n_imputed_upstream` is an ordered
count rather than an indicator, so it is scaled like a continuous feature.


In [5]:
Xf_train, Xf_val, *_ = get_splits(
    include_metadata=True, include_imputation_flags=True, scale_numeric=False)
flag_cols = [c for c in Xf_train.columns if c.endswith('__imputed_upstream')]

from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(Xf_train[flag_cols])
z = sc.transform(Xf_train[flag_cols])

print('effect of standardising a binary flag (train prevalence -> encoded values):')
for i, c in enumerate(flag_cols[:4]):
    p = float(Xf_train[c].mean())
    print(f'  {c.split(" (")[0]:26} prevalence {p:5.1%}  ->  off={z[:,i].min():+.3f}  on={z[:,i].max():+.3f}')
print('\nPassed through unscaled, off stays exactly 0.0 and on is exactly 1.0.')


effect of standardising a binary flag (train prevalence -> encoded values):
  Ammonia                    prevalence 18.2%  ->  off=-0.472  on=+2.119
  Biochemical Oxygen Demand  prevalence 23.6%  ->  off=-0.557  on=+1.797
  Dissolved Oxygen           prevalence 32.3%  ->  off=-0.690  on=+1.449
  Orthophosphate             prevalence 16.5%  ->  off=-0.445  on=+2.247

Passed through unscaled, off stays exactly 0.0 and on is exactly 1.0.


## 4. One-hot or learned embeddings

The work plan asks to test embeddings against one-hot encoding for categorical columns
such as site or station.

**`Area` is not a feature.** It is the grouping key for the site-held-out split, so every
test site is unseen by construction and an embedding for it could never be looked up at
test time. That removes the only high-cardinality candidate.

What remains is low-cardinality, where one-hot is the appropriate default: embeddings
earn their keep by compressing hundreds or thousands of levels, and add parameters and a
tuning axis for no new capacity at this size. `build_nn_preprocessor(categorical='ordinal')`
emits integer codes so the architecture track can test this rather than take it on trust.


In [6]:
for c in ['Country', 'Waterbody Type']:
    print(f'  {c:16} {X_train[c].nunique():>3} levels')
oh = build_nn_preprocessor(X_train.head(50_000), strategy='phase1', categorical='onehot')
od = build_nn_preprocessor(X_train.head(50_000), strategy='phase1', categorical='ordinal')
print(f'\n  one-hot width : {oh.transform(X_train.head(5)).shape[1]} columns')
print(f'  ordinal width : {od.transform(X_train.head(5)).shape[1]} columns (embedding input)')
print(f'  cost of one-hot: {oh.transform(X_train.head(5)).shape[1] - od.transform(X_train.head(5)).shape[1]} extra columns')


  Country            5 levels
  Waterbody Type    12 levels



  one-hot width : 21 columns
  ordinal width : 10 columns (embedding input)
  cost of one-hot: 11 extra columns


## 5. The deciding experiment

Conditioning statistics describe the *inputs*, not the *result*. The work plan is explicit
that Phase 1 stays unless the alternative is clearly better, so the choice is settled by
training an identical MLP under each strategy — same subsample, architecture, seed,
optimiser and epoch budget, scored through the shared `evaluate()`.

Run over three seeds, because a single run cannot distinguish a real gap from noise.


In [7]:
runs = []
for s in (42, 123, 456):
    d = compare_on_mlp(seed=s, verbose=False)
    d['seed'] = s
    runs.append(d)
allr = pd.concat(runs, ignore_index=True)

agg = (allr.groupby('strategy')[['val_mae','val_rmse','val_r2']]
           .agg(['mean','std']).round(4).sort_values(('val_mae','mean')))
display(agg)

p1 = allr[allr.strategy=='phase1'].val_mae
qt = allr[allr.strategy=='quantile'].val_mae
gap, noise = p1.mean()-qt.mean(), max(p1.std(), qt.std())
print(f'phase1   {p1.mean():.4f} +/- {p1.std():.4f}')
print(f'quantile {qt.mean():.4f} +/- {qt.std():.4f}')
print(f'gap {gap:.4f} vs noise {noise:.4f}  ->  {gap/noise:.1f}x the seed-to-seed spread')


val_mae         val_rmse          val_r2        
               mean     std     mean     std    mean     std
strategy                                                    
quantile     0.8015  0.1002   1.2546  0.1024  0.9954  0.0008
robust       1.0440  0.0999   1.6515  0.2841  0.9919  0.0028
yeo-johnson  1.1107  0.0800   2.0565  0.0880  0.9877  0.0010
phase1       1.2348  0.0602   1.7055  0.1215  0.9915  0.0012

phase1   1.2348 +/- 0.0602
quantile 0.8015 +/- 0.1002
gap 0.4333 vs noise 0.1002  ->  4.3x the seed-to-seed spread


## Decision and evidence

| # | Question | Decision | Evidence |
| :-- | :--- | :--- | :--- |
| 1 | Transform for skew? | **Quantile-normal** for the NN; log1p rejected | log1p undefined on 5 of 8 columns (negatives retained by Phase 1 cleaning) and worsens DO −1.39 → −3.17 |
| 2 | Standard or robust? | **Neither** — robust is worst | robust max \|z\| 740 vs standard 48 on validation; IQR division inflates a long tail |
| 3 | Imputation flags? | Binary **passthrough**, count scaled | standardising a 0/1 column makes "absent" non-zero and ties magnitude to train prevalence |
| 4 | One-hot or embeddings? | **One-hot** | `Area` is the split key, not a feature; what remains is 5 and 12 levels, too small for embeddings to pay |

### Does the new pipeline clear the bar?

**Yes.** Over three seeds the quantile-normal pipeline reaches validation MAE **0.80 ± 0.10**
against Phase 1's **1.23 ± 0.06** — a gap of 0.43, more than four times the seed-to-seed
spread, and it also wins on RMSE and $R^2$. The work plan's bar is "clearly better", and a
35 % MAE reduction at four times the noise clears it.

### What this does *not* change

The Phase 1 pipeline is **untouched** for Random Forest and XGBoost. Trees are invariant
to monotone transforms, so re-running them through this would change nothing except the
comparability of the results table. This is an input pipeline for the neural models only.

### The honest caveat

Quantile-normal maps each feature to a normal distribution **by rank**. It preserves
ordering but distorts distance: equal gaps in concentration no longer map to equal gaps in
the input. For a target that is a closed-form function of these measurements that is a real
trade-off, and it is why Yeo-Johnson — monotone *and* distance-preserving — is kept
available as `strategy='yeo-johnson'`. On this evidence it conditions the inputs well
(mean \|skew\| 6.33 → 1.36) but did not convert that into accuracy.

**Handover:** Himanshu's architecture track and my own regularization study (step 2) should
use `build_nn_preprocessor(X_train, strategy='quantile')`, fitted on train only.
